# Прогноз оттока клиентов

## Настройка

In [ ]:
import sys
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "config" / "params.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import plotting
from common import get_engine, load_csv, load_json, load_params, run_sql_file

%matplotlib inline
plotting.use_style()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

params = load_params()
DATA_DIR = ROOT / params["paths"]["data_dir"]
COLORS = plotting.CATEGORICAL

print("корень проекта:", ROOT)
print("конец истории данных:", params["data"]["history_end_date"])
print("random_state:", params["project"]["random_state"])

In [ ]:
TOLERANCE = 1e-9
TOP_N = 15

def check(what, ours, reference):
    ours = np.asarray(ours, dtype=float)
    reference = np.asarray(reference, dtype=float)
    same = ours.shape == reference.shape and np.allclose(
        ours, reference, rtol=TOLERANCE, atol=TOLERANCE, equal_nan=True)
    assert same, f"{what}: расчёт {ours} не совпал с артефактом {reference}"
    print(f"✓ {what} — совпадает с артефактом")

def pct(x, digits=1):
    return f"{x * 100:.{digits}f}%"

def fmt(n):
    return f"{int(n):,}".replace(",", " ")

In [ ]:
try:
    engine = get_engine(params)
    with engine.connect():
        pass
    db_ok = True
except Exception as exc:
    engine, db_ok = None, False
    print("база недоступна:", type(exc).__name__)

required = ["02_episodes_primary.parquet", "02_customer_profile.parquet",
            "02_customer_products.parquet", "04_active_days.parquet",
            "04_registry_audit.parquet", "base.parquet", "07_slice_static.parquet"]
missing = [name for name in required if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"нет выгрузок {missing}: запустите make all при работающей базе")

print("база доступна:", "да" if db_ok else "нет")
print("выгрузки в data/: на месте")

## Данные

### Таблицы базы

In [ ]:
inventory = load_csv("00_tables_inventory.csv")
if db_ok:
    live = run_sql_file(engine, "01_tables_inventory.sql",
                        {"schema": params["db"]["schema"],
                         "table_prefix": params["db"]["table_prefix"]}, params=params)
    live = live.set_index("table_name").reindex(inventory.table_name)
    check("число строк в таблицах", live.n_rows, inventory.n_rows)

profile = load_csv("01_tables_profile.csv")
columns = ["table_name", "purpose", "verdict", "n_rows", "n_customer_ids", "date_min", "date_max"]
display(profile[columns].sort_values("n_rows", ascending=False).reset_index(drop=True))

### Витрина churn_customers_360

In [ ]:
rule = load_csv("01_mart360_churn_rule.csv")
display(rule[["is_churned", "n_customers", "days_min", "days_max"]])

churned = rule[rule.is_churned == 1].iloc[0]
check("порог витрины в конфиге", churned.days_min, params["target"]["mart_churn_days_since_last_txn"])

## Порог тишины

### Эпизоды тишины

In [ ]:
import silence

silence_cfg = params["silence"]
episodes, _ = silence._prepare(params)
print(f"эпизодов тишины: {fmt(len(episodes))}, клиентов: {fmt(episodes.customer_id.nunique())}")
display(episodes[["customer_id", "start_day", "gap_days", "returned", "duration_days"]].head())

### Кривая возврата

In [ ]:
km = silence.kaplan_meier(episodes.duration_days.to_numpy(),
                          episodes.returned.to_numpy(dtype=bool))
horizon = silence_cfg["primary_horizon_days"]
curve = silence.return_curve(episodes, silence_cfg["grid_days"], [horizon],
                             silence_cfg["ci_z"], km=km)
knee = silence.knee_point(curve.silence_days.to_numpy(float), curve.p_return.to_numpy(float))
p_knee = curve.loc[curve.silence_days == knee, "p_return"].iloc[0]

decision = load_json("02_threshold_decision.json")
check("порог тишины (колено кривой)", knee, decision["knee_silence_days"])
check("порог тишины в конфиге", knee, silence_cfg["silence_days"])
check("доля вернувшихся на пороге", p_knee, decision["p_return_at_knee"])
display(curve[["silence_days", "n_eligible_episodes", "p_return", "ci_low", "ci_high"]])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(curve.silence_days, curve.ci_low, curve.ci_high,
                color=COLORS[0], alpha=0.15, linewidth=0)
ax.plot(curve.silence_days, curve.p_return, marker="o", color=COLORS[0])
ax.axvline(knee, color=plotting.NEUTRAL, linestyle="--", linewidth=1)
ax.annotate(f"порог {knee:.0f} дн.: вернулись {pct(p_knee)}", (knee, p_knee),
            xytext=(12, 18), textcoords="offset points", color=plotting.TEXT_PRIMARY)
ax.yaxis.set_major_formatter(PercentFormatter(1))
plotting.tidy(ax, xlabel="клиент уже молчит, дней", ylabel="доля вернувшихся",
              title="Вероятность вернуться после паузы",
              subtitle=f"возврат в течение {horizon} дн.; полоса — доверительный интервал")
plt.show()

## Целевая переменная

### Окна наблюдения, лага и исхода

In [ ]:
target_cfg = params["target"]
T0_TRAIN, T0_OOT = [pd.Timestamp(d).date() for d in target_cfg["snapshot_dates"]]
OBS, LAG, OUTCOME = (target_cfg[k] for k in ("observation_days", "lag_days", "outcome_days"))

windows = load_json("03_target_decision.json")["windows"]
check("длины окон: наблюдение, лаг, исход", [OBS, LAG, OUTCOME],
      [windows["observation_days"], windows["lag_days"], windows["outcome_days"]])
print("значимые продуктовые события:", ", ".join(target_cfg["target_product_event_types"]))
print("вариант метки:", target_cfg["label_variant"])

layout = pd.DataFrame([{
    "снимок": role,
    "начало наблюдения": t0 - timedelta(days=OBS),
    "t0": t0,
    "начало окна исхода": t0 + timedelta(days=LAG),
    "конец окна исхода": t0 + timedelta(days=LAG + OUTCOME),
} for role, t0 in [("train", T0_TRAIN), ("oot", T0_OOT)]])
display(layout)

history_end = pd.Timestamp(params["data"]["history_end_date"]).date()
assert (layout["конец окна исхода"] <= history_end).all(), "окно исхода вылезло за конец истории"
assert layout["конец окна исхода"][0] <= layout["начало окна исхода"][1], "окна исхода пересекаются"
print("✓ окна исхода помещаются в историю и не пересекаются между train и OOT")

### Разметка

In [ ]:
from cohort import COHORT_RULE, load_index
from target import as_frame, mark

activity = load_index(params, "")
marks = as_frame(mark(activity, T0_TRAIN, silence_cfg["silence_days"], LAG, OUTCOME,
                      target_cfg["label_variant"], target_cfg["false_churn_tail_days"],
                      COHORT_RULE))
display(marks[["customer_id", "in_cohort", "days_silent_at_snapshot", "n_ops_in_lag",
               "n_ops_in_label_window", "target"]].head(8))

## База наблюдений

### Каскад отбора

In [ ]:
import cohort

registry = pd.read_parquet(DATA_DIR / "04_registry_audit.parquet")
populations = {role: cohort.build_population(params, activity, t0, registry)
               for role, t0 in [("train", T0_TRAIN), ("oot", T0_OOT)]}
waterfall = pd.concat([cohort.waterfall(params, pop, role) for role, pop in populations.items()])
display(waterfall.pivot_table(index=["step_order", "rule_basis"], columns="snapshot_role",
                              values="n_after", sort=False))
check("каскад отбора: сколько клиентов осталось после каждого шага",
      waterfall.n_after, load_csv("04_cohort_waterfall.csv").n_after)

base_summary = load_json("04_base_summary.json")["snapshots"]
for role, pop in populations.items():
    base = pop[pop.in_base]
    check(f"{role}: размер базы", len(base), base_summary[role]["n_base"])
    check(f"{role}: доля оттока", base.target.mean(), base_summary[role]["churn_rate"])
    print(f"   {role}: {fmt(len(base))} клиентов, ушли {fmt(base.target.sum())} "
          f"({pct(base.target.mean(), 2)})")

### Отток по давности последней операции

In [ ]:
edges = target_cfg["recency_bucket_edges_days"]
labels = [f"{edges[0]}-{edges[1]}"] + [f"{lo + 1}-{hi}" for lo, hi in zip(edges[1:-1], edges[2:])]
train_base = populations["train"].query("in_base")
bucket = pd.cut(train_base.days_silent_at_snapshot, bins=edges, labels=labels, include_lowest=True)
by_recency = (train_base.groupby(bucket, observed=True)
              .agg(clients=("customer_id", "size"), churn_rate=("target", "mean")))
reference = load_csv("04_target_by_recency.csv").query("snapshot_role == 'train'")
check("доля оттока по давности последней операции", by_recency.churn_rate, reference.churn_rate)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(by_recency.index.astype(str), by_recency.churn_rate, color=COLORS[0], width=0.6)
for i, (rate, n) in enumerate(zip(by_recency.churn_rate, by_recency.clients)):
    ax.text(i, rate, f"{pct(rate)}\n{fmt(n)} кл.", ha="center", va="bottom", fontsize=9,
            color=plotting.TEXT_SECONDARY)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_ylim(0, by_recency.churn_rate.max() * 1.3)
plotting.tidy(ax, xlabel="дней без значимой операции на t0", ylabel="доля оттока",
              title="Чем дольше молчит, тем выше риск", subtitle="снимок train")
plt.show()

## Отток во времени

In [ ]:
import dynamics

grid = dynamics.monthly_grid(params)
rows = []
for month in grid.loc[grid.in_extended_series, "month"]:
    pop = cohort.build_population(params, activity, month, registry)
    base = pop[pop.in_base]
    low, high = silence.wilson_ci(int(base.target.sum()), len(base), silence_cfg["ci_z"])
    rows.append({"month": pd.Timestamp(month), "n_base": len(base),
                 "churn_rate": base.target.mean(), "ci_low": low, "ci_high": high})
monthly = pd.DataFrame(rows)

reference = load_csv("05_monthly_churn.csv")
check("размер базы по месяцам", monthly.n_base, reference.n_base)
check("доля оттока по месяцам", monthly.churn_rate, reference.churn_rate)

In [ ]:
dyn = load_json("05_dynamics_decision.json")
tariff_date = pd.Timestamp(dyn["event_identification"]["tariff_effective_date_in_data"])
brk = dyn["break"]

fig, ax = plt.subplots(figsize=(9, 4))
ax.fill_between(monthly.month, monthly.ci_low, monthly.ci_high, color=COLORS[0], alpha=0.15,
                linewidth=0)
ax.plot(monthly.month, monthly.churn_rate, marker="o", color=COLORS[0])
ax.axvline(tariff_date, color=plotting.DIVERGING_LOW, linestyle="--", linewidth=1)
ax.annotate("пересмотр тарифа", (tariff_date, monthly.churn_rate.max()), xytext=(6, 0),
            textcoords="offset points", color=plotting.DIVERGING_LOW)
for role, t0 in [("train", T0_TRAIN), ("OOT", T0_OOT)]:
    ax.axvline(pd.Timestamp(t0), color=plotting.NEUTRAL, linewidth=1, linestyle=":")
    ax.annotate(role, (pd.Timestamp(t0), monthly.churn_rate.min()), xytext=(4, 0),
                textcoords="offset points", color=plotting.TEXT_SECONDARY)
ax.yaxis.set_major_formatter(PercentFormatter(1))
plotting.tidy(ax, xlabel="дата снимка t0", ylabel="доля оттока",
              title="Доля оттока по месяцам", subtitle="полоса — доверительный интервал")
plt.show()

print(f"точка разрыва: {brk['break_at']}; доля оттока до: {pct(brk['rate_before'], 2)}, после: {pct(brk['rate_after'], 2)}; пересмотр тарифа: {tariff_date.date()}")

## Признаки

### Матрица признаков

In [ ]:
import feature_store as FS

matrices = {role: FS.feature_matrix(params, t0) for role, t0 in [("train", T0_TRAIN), ("oot", T0_OOT)]}
store_ref = load_json("07_store_build.json")
for role, matrix in matrices.items():
    assert FS.matrix_hash(matrix) == store_ref[role]["hash"], f"{role}: витрина разошлась"
    print(f"✓ {role}: {fmt(len(matrix))} клиентов × {matrix.shape[1] - 2} признаков — "
          "совпадает с витриной пайплайна побайтно")
display(matrices["train"].iloc[:5, :10])

### Проверка признака вручную

In [ ]:
matrix = matrices["train"]
k = min(w for w in params["features"]["windows_days"] if f"txn_cnt_{w}d" in matrix.columns)
txn = FS.source_slices(params, T0_TRAIN)["txn"]
age = (pd.Timestamp(T0_TRAIN) - pd.to_datetime(txn.event_day)).dt.days
recent = txn[(txn.status == target_cfg["significant_operation_status"]) & (age < k)]
manual = recent.groupby("customer_id").size().reindex(matrix.customer_id, fill_value=0)
check(f"txn_cnt_{k}d, посчитанный вручную", manual.to_numpy(), matrix[f"txn_cnt_{k}d"].to_numpy())

### Отбор признаков

In [ ]:
from modeling import data as D

meta = D.feature_meta(params)
summary = load_json("06_features_summary.json")
blocks = pd.DataFrame([{"блок": b, "тема": v["title"], "построено": v["n_features"],
                        "включено": v["n_included"]} for b, v in summary["by_block"].items()])
display(blocks)

features = D.feature_variants(params)["full"]
check("число отобранных признаков", len(features), summary["n_included"])
rejected = meta.loc[meta.verdict != "ВКЛЮЧЕНА", "verdict_reason"]
reasons = np.where(rejected.str.startswith("дублирует"), "дубль другого признака",
                   "сила неотличима от случайной")
print(f"построено {summary['n_features']}, отобрано {len(features)}, отклонено {len(rejected)}:")
print(pd.Series(reasons).value_counts().to_string())

### Сила признаков по отдельности

In [ ]:
from features import screening

train = D.dataset(params, D.ROLE_TRAIN, T0_TRAIN, ix=activity)
oot = D.dataset(params, D.ROLE_OOT, T0_OOT, ix=activity)

auc = pd.Series({f: screening.auc(train.X[f].to_numpy(float), train.y) for f in features})
check("AUC каждого признака на train", auc, meta.set_index("feature").auc_train[features])

RANDOM_AUC = 0.5
strength = (auc - RANDOM_AUC).abs().sort_values(ascending=False).head(TOP_N)[::-1]
fig, ax = plt.subplots(figsize=(8, 5.5))
colors = [plotting.DIVERGING_LOW if auc[f] > RANDOM_AUC else plotting.DIVERGING_HIGH
          for f in strength.index]
ax.barh(strength.index, auc[strength.index] - RANDOM_AUC, color=colors, height=0.6)
ax.axvline(0, color=plotting.NEUTRAL, linewidth=1)
plotting.tidy(ax, xlabel="AUC признака минус середина шкалы",
              title="Самые сильные признаки по отдельности",
              subtitle="красный — больше значение, выше риск; синий — наоборот")
plt.show()

### Мультиколлинеарность

In [ ]:
from modeling import importance as I

correlation = I.max_correlation_per_feature(train, features)
check("максимальная корреляция каждого признака с остальными",
      correlation.set_index("feature").max_abs_spearman.sort_index(),
      load_csv("08_feature_correlation.csv").set_index("feature").max_abs_spearman.sort_index())
print(f"самая сильная связь между отобранными признаками: |r| = "
      f"{correlation.max_abs_spearman.max():.3f}, порог отсева — {params['features']['corr_threshold']}")

## Проверка на утечку

### Даты в срезах

In [ ]:
rows = []
for role, t0 in [("train", T0_TRAIN), ("oot", T0_OOT)]:
    for source, frame in FS.source_slices(params, t0).items():
        date_columns = [c for c in ("event_day", "opened_at", "closed_at") if c in frame.columns]
        for column in date_columns:
            latest = pd.to_datetime(frame[column]).max()
            rows.append({"снимок": role, "срез": source, "колонка": column,
                         "последняя дата": latest.date(), "t0": t0,
                         "не позже t0": latest.date() <= t0})
dates = pd.DataFrame(rows)
display(dates)
assert dates["не позже t0"].all(), "в срезах есть записи из будущего"
print("✓ ни в одном срезе нет записей позже t0")

### Автотесты пайплайна

In [ ]:
tests = load_csv("07_test_results.csv")
display(tests[["test", "question", "n_checks", "n_failed", "passed"]])
shuffle = load_json("07_target_shuffle.json")
print(f"пройдено проверок: {int(tests.passed.sum())} из {len(tests)}")
print(f"AUC на перемешанной метке: {shuffle['null_ci_low']:.3f}–{shuffle['null_ci_high']:.3f}, на настоящей: {shuffle['auc_real_target']:.3f}")

## Обучение моделей

### Разбиение по времени

In [ ]:
plan = D.snapshot_plan(params)
D.check_plan(params, plan)
holdout_plan = D.holdout_plan(params)
display(pd.concat([plan[["role", "t0", "purpose", "outcome_window_end"]],
                   pd.DataFrame([{"role": "holdout", "t0": holdout_plan["t0"],
                                  "purpose": "заморожен до этапа 12",
                                  "outcome_window_end": holdout_plan["outcome_window_end"]}])],
                  ignore_index=True))

roles = dict(zip(plan.role, plan.t0))
inner = [D.dataset(params, r, roles[r], ix=activity) for r in plan.role if r.startswith("inner_")]
reconciliation = D.reconcile_with_base(params, {"train": train, "oot": oot})
assert reconciliation.passed.all(), "метки разошлись с базой этапа 4"
print("✓ состав клиентов и метки train/OOT совпадают с базой наблюдений этапа 4")
for ds in [*inner, train, oot]:
    print(f"   {ds.role:8s} t0={ds.t0}: {fmt(ds.n)} клиентов, доля оттока {pct(ds.prevalence, 2)}")

### Наивные бейзлайны

In [ ]:
from modeling import baselines as B
from modeling import metrics as M

fractions = params["model"]["topk_fractions"]
scores = dict(B.scores(params, oot))
probas = dict(B.probabilities(params, oot))
naive = pd.DataFrame([{"правило": name, "ROC-AUC": M.roc_auc(oot.y, s), "PR-AUC": M.pr_auc(oot.y, s)}
                      for name, s in scores.items()])
display(naive)
reference = load_csv("08_model_comparison.csv").set_index("model")
check("PR-AUC наивных правил на OOT", naive["PR-AUC"], reference.loc[naive["правило"], "pr_auc"])

### Подбор гиперпараметров

In [ ]:
import json
from modeling import estimators as E

manifest = json.loads((ROOT / "models" / "manifest.json").read_text(encoding="utf-8"))
search_ref = load_csv("08_hyperparam_search.csv")

folds = list(zip([*inner, train][:-1], [*inner, train][1:]))
for fit_ds, val_ds in folds:
    print(f"фолд: обучение {fit_ds.t0} → проверка {val_ds.t0}")

RUN_LOGIT_SEARCH = True
if RUN_LOGIT_SEARCH:
    logit_trials = E.walk_forward_search(params, "logit", folds, features)
    check("PR-AUC всех проб логистической регрессии",
          logit_trials.pr_auc_mean, search_ref.query("family == 'logit'").pr_auc_mean)
else:
    logit_trials = search_ref.query("family == 'logit'")

best = {}
for family in E.FAMILIES:
    trials = logit_trials if family == "logit" else search_ref.query("family == @family")
    row = trials.sort_values(["pr_auc_mean", "draw"], ascending=[False, True]).iloc[0]
    best[family] = {"проб": len(trials), "class_weight": row.class_weight,
                    "PR-AUC на фолдах": row.pr_auc_mean, "гиперпараметры": row.hyperparams}
display(pd.DataFrame(best).T)

for family in E.FAMILIES:
    weight = manifest["models"][family]["class_weight"]
    assert best[family]["class_weight"] == ("none" if weight is None else weight)
print("✓ лучшие пробы совпадают с гиперпараметрами в models/manifest.json")

### Финальные модели на OOT

In [ ]:
models = {}
for family in E.FAMILIES:
    chosen = manifest["models"][family]
    models[family] = E.fit(E.build(params, family, chosen["hyperparams"], chosen["class_weight"]),
                           train, features)
    scores[family] = probas[family] = E.predict(models[family], oot, features)
    print(f"обучена: {E.FAMILY_TITLE[family]}")

seed = params["project"]["random_state"]
level = params["features"]["auc_ci_level"]
point = {name: M.point_metrics(oot.y, s, probas.get(name), fractions) for name, s in scores.items()}
draws = M.bootstrap_draws(oot.y, scores, probas,
                          M.bootstrap_indices(oot.n, params["features"]["auc_bootstrap_repeats"], seed))
comparison = M.comparison_table(point, draws, level).set_index("model")

shown = ["roc_auc", "roc_auc_ci_low", "roc_auc_ci_high", "pr_auc", "pr_auc_ci_low",
         "pr_auc_ci_high", "pr_auc_over_prevalence", "brier"]
display(comparison[shown])
check("метрики и интервалы всех моделей на OOT", comparison[shown],
      reference.loc[comparison.index, shown])

### Выбор модели

In [ ]:
families = list(E.FAMILIES)
significance = M.pairwise_significance({f: draws[f] for f in families},
                                       {f: point[f] for f in families}, level)
pr = significance.query("metric == 'pr_auc'")
display(pr[["model_a", "model_b", "diff", "diff_ci_low", "diff_ci_high", "covers_zero"]])

simplest = families[0]
best_family = max(families, key=lambda f: point[f]["pr_auc"])
pair = pr[pr.model_a.isin([simplest, best_family]) & pr.model_b.isin([simplest, best_family])]
upgrade = best_family != simplest and not pair.covers_zero.iloc[0]
final_family = best_family if upgrade else simplest
assert final_family == manifest["final_model"], "правило выбора дало другую модель, чем пайплайн"
print(f"✓ финальная модель: {E.FAMILY_TITLE[final_family]} (совпадает с манифестом)")

print(f"лучший PR-AUC: {E.FAMILY_TITLE[best_family]}; прирост над логистической регрессией значим: {'да' if upgrade else 'нет'}")

### Кривые PR и ROC

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

model_proba = scores[final_family]
curves_for = [("logit", COLORS[0]), ("forest", COLORS[1]), ("boosting", COLORS[2]),
              ("naive_recency", plotting.NEUTRAL)]
fig, (ax_pr, ax_roc) = plt.subplots(1, 2, figsize=(11, 4.2))
for name, color in curves_for:
    precision, recall, _ = precision_recall_curve(oot.y, scores[name])
    fpr, tpr, _ = roc_curve(oot.y, scores[name])
    title = E.FAMILY_TITLE.get(name, "правило давности")
    ax_pr.plot(recall, precision, color=color, linewidth=1.5,
               label=f"{title}: {point[name]['pr_auc']:.3f}")
    ax_roc.plot(fpr, tpr, color=color, linewidth=1.5,
                label=f"{title}: {point[name]['roc_auc']:.3f}")
ax_pr.axhline(oot.prevalence, color=plotting.NEUTRAL, linestyle=":", linewidth=1)
ax_roc.plot([0, 1], [0, 1], color=plotting.NEUTRAL, linestyle=":", linewidth=1)
plotting.tidy(ax_pr, xlabel="полнота (recall)", ylabel="точность (precision)",
              title="Precision-Recall на OOT", subtitle="в легенде — PR-AUC")
plotting.tidy(ax_roc, xlabel="доля ложных тревог", ylabel="доля найденных ушедших",
              title="ROC на OOT", subtitle="в легенде — ROC-AUC")
ax_pr.legend(fontsize=8)
ax_roc.legend(fontsize=8, loc="lower right")
plt.show()

### Верхушка рейтинга

In [ ]:
topk = pd.concat([M.topk_table(oot.y, scores[name], fractions).assign(model=name)
                  for name in [final_family, "naive_recency"]])
display(topk.pivot_table(index="top_fraction", columns="model",
                         values=["precision", "recall", "lift"], sort=False))
topk_ref = load_csv("08_topk.csv").set_index(["model", "top_fraction"])
check("точность и полнота на верхушке рейтинга",
      topk.set_index(["model", "top_fraction"])[["precision", "recall"]],
      topk_ref.loc[topk.set_index(["model", "top_fraction"]).index, ["precision", "recall"]])

### Калибровка

In [ ]:
n_bins = M.n_bins_by_cell_size(oot.n, params["silence"]["min_episodes_per_cell"])
calib = M.calibration_table(oot.y, model_proba, n_bins)
print("способ калибровки в манифесте:", manifest["calibration"]["method"])
print(f"Brier: {M.brier(oot.y, model_proba):.4f}, средняя ошибка калибровки: "
      f"{M.calibration_error(calib):.4f}")

fig, ax = plt.subplots(figsize=(5, 5))
limit = max(calib.pred_mean.max(), calib.observed_rate.max()) * 1.05
ax.plot([0, limit], [0, limit], color=plotting.NEUTRAL, linestyle=":", linewidth=1)
ax.plot(calib.pred_mean, calib.observed_rate, marker="o", color=COLORS[0])
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.yaxis.set_major_formatter(PercentFormatter(1))
plotting.tidy(ax, xlabel="предсказанная вероятность", ylabel="фактическая доля оттока",
              title="Калибровка на OOT", subtitle="пунктир — идеальная калибровка")
plt.show()

### Стабильность во времени

In [ ]:
stability = []
for role in [r for r in plan.role if r.startswith("stability_")]:
    ds = D.dataset(params, role, roles[role], ix=activity)
    p = E.predict(models[final_family], ds, features)
    stability.append({"t0": str(ds.t0), "клиентов": ds.n, "доля оттока": ds.prevalence,
                      "ROC-AUC": M.roc_auc(ds.y, p), "PR-AUC": M.pr_auc(ds.y, p)})
stability = pd.DataFrame([{"t0": str(oot.t0), "клиентов": oot.n, "доля оттока": oot.prevalence,
                           "ROC-AUC": point[final_family]["roc_auc"],
                           "PR-AUC": point[final_family]["pr_auc"]}, *stability])
display(stability)
check("ROC-AUC по месяцам после OOT", stability["ROC-AUC"], load_csv("08_monthly_oot.csv").roc_auc)

## Анализ признаков

### Коэффициенты логистической регрессии

In [ ]:
from business import interpretation as INT

logit = models[final_family]
coefficients = pd.Series(np.ravel(logit.named_steps["model"].coef_),
                         index=logit[:-1].get_feature_names_out())
nonzero = coefficients[coefficients != 0]
print(f"коэффициентов: {len(coefficients)} (признаки и флаги пропусков), "
      f"обнулено штрафом L1: {(coefficients == 0).sum()}")
top_coef = nonzero.reindex(nonzero.abs().sort_values(ascending=False).index).head(TOP_N)
display(top_coef.rename("коэффициент").to_frame())

### Permutation importance и вклады признаков

In [ ]:
permutation = I.permutation_table(params, logit, oot, features, "average_precision")
perm_ref = load_csv("08_permutation_importance.csv").set_index("feature")
check("permutation importance", permutation.set_index("feature").importance_mean,
      perm_ref.importance_mean.reindex(permutation.feature))

contributions, base_value, controls = INT.contribution_matrix(logit, oot.X[features], train.X[features])
global_importance = INT.global_importance(contributions)
shap_ref = load_csv("09_shap_importance.csv").set_index("feature")
check("средний модуль вклада признака", global_importance.set_index("feature").mean_abs_shap,
      shap_ref.mean_abs_shap.reindex(global_importance.feature))
print(f"разложение точное: невязка с прогнозом модели {controls['max_identity_error']:.1e}")

fig, (ax_p, ax_s) = plt.subplots(1, 2, figsize=(12, 5.5))
top_p = permutation.head(TOP_N)[::-1]
ax_p.barh(top_p.feature, top_p.importance_mean, xerr=top_p.importance_std, color=COLORS[0],
          height=0.6, error_kw={"ecolor": plotting.TEXT_SECONDARY, "linewidth": 1})
plotting.tidy(ax_p, xlabel="падение PR-AUC при перемешивании", title="Permutation importance")
top_s = global_importance.head(TOP_N)[::-1]
ax_s.barh(top_s.feature, top_s.share_of_total, color=COLORS[0], height=0.6)
ax_s.xaxis.set_major_formatter(PercentFormatter(1))
plotting.tidy(ax_s, xlabel="доля в сумме вкладов", title="Средний вклад в прогноз")
fig.tight_layout()
plt.show()

### Наборы признаков без рискованных

In [ ]:
chosen = manifest["models"][final_family]
variant_rows = []
for name, columns in D.feature_variants(params).items():
    model = E.fit(E.build(params, final_family, chosen["hyperparams"], chosen["class_weight"]),
                  train, columns)
    p = model.predict_proba(oot.X[columns])[:, 1]
    variant_rows.append({"набор": name, "признаков": len(columns),
                         "ROC-AUC": M.roc_auc(oot.y, p), "PR-AUC": M.pr_auc(oot.y, p)})
variants = pd.DataFrame(variant_rows)
display(variants)
variants_ref = load_csv("08_feature_set_variants.csv").query("family == @final_family")
check("качество моделей на разных наборах признаков", variants["PR-AUC"], variants_ref.pr_auc)

### Разбор одного клиента

In [ ]:
y = pd.Series(oot.y, index=oot.X.index)
proba = pd.Series(model_proba, index=oot.X.index)
business_cfg = params["business"]
picks = INT.pick_examples(proba, y, business_cfg["top_share"])
values_seen = INT.transformed(logit, oot.X[features])
examples = INT.local_examples(contributions, values_seen, proba, y, picks,
                              top_k=business_cfg["local_example_features"])
role = next(iter(picks))
example = examples[examples.role == role]
print(f"{role}: клиент {example.customer_id.iloc[0]}, вероятность ухода "
      f"{pct(example.proba.iloc[0])}, фактически ушёл: {'да' if example.target.iloc[0] else 'нет'}")
display(example[["feature", "value", "shap", "direction"]].rename(
    columns={"value": "значение (стандартизованное)", "shap": "вклад в логит"}))

## Применение для бизнеса

### Зона риска

In [ ]:
from business import economics as EC
from business import segments as S

top_share = business_cfg["top_share"]
mask = S.top_mask(proba, top_share)
business_ref = load_json("09_business.json")
check("доля оттока в зоне риска", y[mask].mean(), business_ref["risk_zone_churn_rate"])

versus, overlap = S.versus_naive(proba, pd.Series(scores["naive_recency"], index=y.index),
                                 y, top_share)
display(versus)
check("модель против правила давности", versus["из них ушедших"],
      load_csv("09_versus_naive.csv")["из них ушедших"])
print(f"зона риска: {fmt(mask.sum())} клиентов, доля оттока {pct(y[mask].mean())}, lift {y[mask].mean() / y.mean():.1f}")

### Сегменты зоны риска

In [ ]:
reachable = S.reachable_mask(oot.meta, oot.X, business_cfg["silent_edge_days"],
                             business_cfg["app_active_window_days"])
assigned = S.assign(contributions, meta, mask, reachable)
components = EC.value_components(oot.X)
segments = S.summarise(assigned, y, proba, components.observed_revenue_monthly,
                       silence_cfg["ci_z"])
check("размеры сегментов зоны риска", segments["клиентов"], load_csv("09_segments.csv")["клиентов"])
display(segments[["сегмент", "клиентов", "фактическая доля оттока", "нижняя граница доли",
                  "верхняя граница доли", "рекомендованное действие"]])

### Экономика удержания

In [ ]:
value = EC.horizon_value(components, business_cfg["interchange_rate_grid"][0], OUTCOME)
anchor = float(components.observed_revenue_monthly[mask.to_numpy()].mean())
budget = EC.budget_curve(proba, value, y, business_cfg["budget_share_grid"],
                         business_cfg["uplift_grid"], business_cfg["contact_cost_ratios"], anchor)
optimum = EC.optimum(budget)
check("оптимальный чистый эффект по сценариям", optimum.expected_net,
      load_csv("09_budget_optimum.csv").expected_net)
display(optimum[["cost_ratio", "uplift", "share", "contacted", "expected_saved",
                 "actual_churners_in_list", "expected_net", "roi", "profitable"]])

capacity = load_csv("09_capacity.csv").iloc[0]
print(f"прибыльных сценариев: {int(optimum.profitable.sum())} из {len(optimum)}")
print(f"дней на обзвон зоны риска при текущем темпе: {capacity['дней на обзвон зоны риска при текущем темпе']:.0f}, лаг: {LAG}")

## Финальная проверка на холдауте

In [ ]:
holdout = D.dataset(params, "holdout", holdout_plan["t0"], ix=activity)
holdout_proba = E.predict(logit, holdout, features)
holdout_naive = B.scores(params, holdout)["naive_recency"]
holdout_ref = load_json("12_holdout_metrics.json")

result = pd.DataFrame([
    {"снимок": "OOT", "t0": str(oot.t0), "клиентов": oot.n, "доля оттока": oot.prevalence,
     "ROC-AUC": point[final_family]["roc_auc"], "PR-AUC": point[final_family]["pr_auc"],
     "PR-AUC правила давности": point["naive_recency"]["pr_auc"]},
    {"снимок": "холдаут", "t0": str(holdout.t0), "клиентов": holdout.n,
     "доля оттока": holdout.prevalence, "ROC-AUC": M.roc_auc(holdout.y, holdout_proba),
     "PR-AUC": M.pr_auc(holdout.y, holdout_proba),
     "PR-AUC правила давности": M.pr_auc(holdout.y, holdout_naive)},
])
display(result)
h = result.iloc[1]
check("метрики на холдауте", [h["ROC-AUC"], h["PR-AUC"], h["доля оттока"]],
      [holdout_ref["roc_auc"], holdout_ref["pr_auc"], holdout_ref["prevalence"]])

## Итоги

In [ ]:
final = pd.DataFrame([
    ("порог тишины, дней", f"{knee:.0f}", "02_threshold_decision.json"),
    ("окна: наблюдение / лаг / исход, дней", f"{OBS} / {LAG} / {OUTCOME}", "03_target_decision.json"),
    ("база train / OOT, клиентов", f"{fmt(train.n)} / {fmt(oot.n)}", "04_base_summary.json"),
    ("доля оттока train / OOT", f"{pct(train.prevalence, 2)} / {pct(oot.prevalence, 2)}",
     "04_base_summary.json"),
    ("признаков построено / в модели", f"{summary['n_features']} / {len(features)}",
     "06_features_summary.json"),
    ("финальная модель", E.FAMILY_TITLE[final_family], "08_metrics.json"),
    ("ROC-AUC / PR-AUC на OOT", f"{point[final_family]['roc_auc']:.3f} / "
     f"{point[final_family]['pr_auc']:.3f}", "08_model_comparison.csv"),
    ("PR-AUC правила давности на OOT", f"{point['naive_recency']['pr_auc']:.3f}",
     "08_model_comparison.csv"),
    ("доля оттока в зоне риска", pct(y[mask].mean()), "09_business.json"),
    ("ROC-AUC / PR-AUC на холдауте", f"{h['ROC-AUC']:.3f} / {h['PR-AUC']:.3f}",
     "12_holdout_metrics.json"),
], columns=["что", "значение", "артефакт-источник"])
display(final)